In [1]:
!pip install fastapi uvicorn transformers accelerate sentencepiece bitsandbytes


In [2]:
%%writefile ls_backend_7b.py
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ---------------------------------------------------------
# Load the model (quantized recommended for Colab)
# ---------------------------------------------------------
MODEL_NAME = "unsloth/llama-3-8b-Instruct-bnb-4bit"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",           # Uses GPU if available
    torch_dtype=torch.float16
)

model.eval()

LABELS = ["INFORMATIVE", "FIGURE_RELATED", "OTHER"]

# ---------------------------------------------------------
# FastAPI app
# ---------------------------------------------------------
app = FastAPI()

class PredictRequest(BaseModel):
    data: list   # [{"text": "..."}]


def classify(text: str) -> str:
    """Prompt-based classification using the 7B model"""
    prompt = f"""
Classify the following patent sentence into exactly one category:
- INFORMATIVE
- FIGURE_RELATED
- OTHER

Sentence: "{text}"

Return only the label.
""".strip()

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=5,
            pad_token_id=tokenizer.eos_token_id,
        )

    answer = tokenizer.decode(output[0], skip_special_tokens=True).strip()

    # Extract the label robustly
    for label in LABELS:
        if label.lower() in answer.lower():
            return label

    return "OTHER"  # fallback


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(request: PredictRequest):
    results = []
    for item in request.data:
        text = item["text"]
        pred = classify(text)

        results.append({
            "result": [
                {
                    "from_name": "label",
                    "to_name": "text",
                    "type": "choices",
                    "value": {"choices": [pred]}
                }
            ],
            "score": 1.0
        })
    return results


Writing ls_backend_7b.py


In [ ]:
# Start FastAPI server on port 8000
import subprocess
server = subprocess.Popen(["uvicorn", "ls_backend_7b:app", "--host", "0.0.0.0", "--port", "8000"])


In [ ]:
from google.colab import output

public_url = output.eval_js("google.colab.kernel.proxyPort(8000)")
public_url
